# Load CSV File into Oracle

## Create Table

In [5]:
import pandas as pd
import os

# ==========================
# 1. Load CSV files
# ==========================
movies_df = pd.read_csv("../data/clean/movies_clean.csv")
boxoffice_df = pd.read_csv("../data/clean/boxoffice_clean.csv")
rt_df = pd.read_csv("../data/clean/rt_reviews_clean.csv")

# ==========================
# 2. CREATE TABLE statements
# ==========================
create_sql = """
-- Drop tables if they exist
BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE rt_reviews PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE box_office PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE movies PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

-- Drop tables if they exist
BEGIN
   EXECUTE IMMEDIATE 'DROP TABLE movie_genres PURGE';
EXCEPTION
   WHEN OTHERS THEN
      IF SQLCODE != -942 THEN
         RAISE;
      END IF;
END;
/

-- Create movies table
CREATE TABLE movies (
    tconst VARCHAR2(20) PRIMARY KEY,
    primary_title VARCHAR2(4000) NOT NULL,
    norm_title VARCHAR2(4000) NOT NULL,
    title_type VARCHAR2(50),
    start_year NUMBER(4),
    runtime_minutes NUMBER,
    imdb_rating NUMBER(3,1)
);

-- Movie genres table (1-to-many relationship)
CREATE TABLE movie_genres (
    tconst VARCHAR2(20) NOT NULL,
    genre VARCHAR2(100) NOT NULL,
    FOREIGN KEY (tconst) REFERENCES movies(tconst) ON DELETE CASCADE
);

-- Create box_office table
CREATE TABLE box_office (
    norm_title VARCHAR2(4000) NOT NULL,
    year NUMBER(4) NOT NULL,
    worldwide_revenue NUMBER,
    PRIMARY KEY (norm_title, year)
);

-- Create rt_reviews table
CREATE TABLE rt_reviews (
    norm_title VARCHAR2(4000) NOT NULL,
    year NUMBER(4) NOT NULL,
    critics_consensus VARCHAR2(4000) NOT NULL,
    review_date DATE NOT NULL,
    tomatometer_rating NUMBER,
    PRIMARY KEY (norm_title, year, review_date)
);
"""

with open("../sql/create_tables.sql", "w") as f:
    f.write(create_sql)

## Insert movies table

In [11]:
# Load movies csv file into movies table
import pandas as pd

# Load CSV
df = pd.read_csv("../data/clean/movies_clean.csv")

table_name = "movies"
sql_lines = []


def safe_int(value):
    """Convert numeric string to int or return NULL if missing."""
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return int(value)

def safe_float(value):
    """Convert numeric string to float or return NULL if missing."""
    if pd.isna(value) or value == "\\N":
        return "NULL"
    return float(value)

for _, row in df.iterrows():
    # Handle numbers safely
    start_year = safe_int(row.get("startYear"))
    runtime = safe_int(row.get("runtimeMinutes"))
    imdb_rating = safe_float(row.get("averageRating"))
    tomatometer = safe_float(row.get("tomatometer"))

    # Handle strings and escape quotes
    primary_title = row["primaryTitle"].replace("'", "''") if pd.notna(row["primaryTitle"]) else ""
    norm_title = row["clean_title"].replace("'", "''") if pd.notna(row.get("clean_title")) else ""
    title_type = row["titleType"].replace("'", "''") if pd.notna(row["titleType"]) else ""

    sql_line = f"""INSERT INTO {table_name} (
    tconst, primary_title, norm_title, title_type, start_year, runtime_minutes, imdb_rating
) VALUES (
    '{row["tconst"]}', '{primary_title}', '{norm_title}', '{title_type}', {start_year}, {runtime}, {imdb_rating}
);"""
    sql_lines.append(sql_line)

sql_lines.append("\nCOMMIT;")

# Save SQL file
with open("../sql/movies_inserts.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))

print("movies_inserts.sql generated successfully!")

movies_inserts.sql generated successfully!


## Insert movie_genres table

In [3]:
df = pd.read_csv("../data/clean/movies_clean.csv")

sql_lines = []

for _, row in df.iterrows():

    if pd.isna(row["genres"]) or row["genres"] == "\\N":
        continue

    genres = row["genres"].split(",")

    for g in genres:

        genre = g.strip().replace("'", "''")

        sql = f"""
INSERT INTO movie_genres (tconst, genre)
VALUES ('{row["tconst"]}', '{genre}');
"""

        sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/movie_genres_insert.sql", "w") as f:
    f.write("\n".join(sql_lines))

## Insert box_office table

In [15]:
df = pd.read_csv("../data/clean/boxoffice_clean.csv")

sql_lines = []

for _, row in df.iterrows():

    revenue = "NULL" if pd.isna(row["$Worldwide"]) else int(row["$Worldwide"])

    title = str(row["clean_title"]).replace("'", "''")

    sql = f"""
INSERT INTO box_office (
    norm_title, year, worldwide_revenue
) VALUES (
    '{title}',
    {int(row["Year"])},
    {revenue}
);
"""

    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/boxoffice_insert.sql", "w") as f:
    f.write("\n".join(sql_lines))

## Insert rt_reviews table

In [16]:
df = pd.read_csv("../data/clean/rt_reviews_clean.csv")

sql_lines = []

for _, row in df.iterrows():

    title = str(row["norm_title"]).replace("'", "''")
    consensus = str(row["critics_consensus"]).replace("'", "''")
    tomatometer_rating = "NULL" if pd.isna(row["tomatometer_rating"]) else row["tomatometer_rating"]

    sql = f"""
INSERT INTO rt_reviews (
    norm_title, year, critics_consensus, review_date, tomatometer_rating
) VALUES (
    '{title}',
    {int(row["year"])},
    '{consensus}',
    TO_DATE('{row["review_date"]}','YYYY-MM-DD'),
    {tomatometer_rating}
);
"""

    sql_lines.append(sql)

sql_lines.append("COMMIT;")

with open("../sql/rt_reviews_insert.sql", "w") as f:
    f.write("\n".join(sql_lines))